# Distributed VLA Fine-Tuning with Ray Train

## TLDR

You take the streaming LIBERO pipeline from notebook 01 and hand it to **Ray
Train** to fine-tune **PI0.5:** a 3.4-billion-parameter Vision-Language-Action
policy across every GPU in the cluster with distributed data-parallel
**(DDP)**. Ray Train wraps the model for
DDP, shards the streaming dataset across workers, reports
synced metrics, and writes a fault-tolerant checkpoint. You write a normal
PyTorch training loop and Ray handles the distribution. Scaling from 2 GPUs to 400
is one number in `ScalingConfig`.

## Introduction

PI0.5 is a flow-matching VLA: a frozen PaliGemma vision-language backbone plus a
small **action expert** that turns the fused representation into robot action
chunks. Fine-tuning the whole 3.4B model on every demonstration is expensive and can be
unnecessary. As such, here we **freeze the backbone and train only the four action-head
projections** (`action_in_proj`, `action_out_proj`, `time_mlp_in`,
`time_mlp_out`). That keeps the trainable footprint tiny while still adapting the
policy for these tutorials.

The interesting part is not the model but rather the fact that the **training loop is a
plain PyTorch loop**, and Ray Train turns it into a synchronized 4-GPU DDP job
with checkpointing and fault tolerance, with no distributed boilerplate.

## Key concepts used in this notebook

**Ray Train** is Ray's distributed-training library. You pass a
`train_loop_per_worker` function to a `TorchTrainer`; Ray launches one copy per
GPU worker, each running your loop on its own shard of the data.

**`prepare_model()`** wraps your `nn.Module` in PyTorch DDP and moves it to the
worker's GPU, replacing manual `init_process_group` / `DistributedDataParallel`.

**`get_dataset_shard()`** hands each worker its slice of the Ray Data stream,
replacing `DistributedSampler` and manual sharding.

**`train.report(metrics, checkpoint=...)`** reports metrics from every worker
(aggregated by Ray) and persists a checkpoint from rank 0.

**`ScalingConfig(num_workers=N, use_gpu=True)`** is the one knob that sets how
many GPU workers run. 4 here; 400+ in production. It's the same training loop.

**`FailureConfig(max_failures=1)`** restarts the job from the last checkpoint on
a worker failure. This is essential for multi-hour runs.

## What you will learn

- Stage a 3.4B model to per-node local disk **once per node** with Ray
- Write a `train_loop_per_worker` and launch it on every GPU in the cluster with `TorchTrainer`
- Use `prepare_model`, `get_dataset_shard`, and `train.report` instead of
  distributed boilerplate
- Freeze a backbone and train only the action heads
- Produce a fault-tolerant checkpoint on shared storage that notebook 03 serves

## Why Ray Train?

| Challenge | Without Ray Train | With Ray Train |
|---|---|---|
| Multi-GPU DDP | `init_process_group`, device juggling, `DDP(...)` | `prepare_model(policy)` |
| Data sharding | `DistributedSampler`, manual splits | `get_dataset_shard("train")` |
| Checkpoint coordination | custom rank-0 filesystem logic | `train.report(checkpoint=...)` |
| Fault tolerance | custom retry/restore logic | `FailureConfig(max_failures=1)` |
| Scale 2 → 400 GPUs | rewrite launch + data plumbing | `ScalingConfig(num_workers=400)` |

## Architecture

```
   Ray Data stream (from notebook 01)
            │  get_dataset_shard("train")  → 2 shards
            ▼
  ┌──────────┬──────────┐
  │ worker 0 │ worker 1 │   each: 1 GPU (≥16 GB)
  │ PI0.5    │ PI0.5    │   prepare_model → DDP
  │ (frozen  │          │   train action heads only
  │  backbone│  …       │   batch 1 × grad-accum 16
  │  + heads)│          │
  └────┬─────┴────┬─────┘
       └── DDP all-reduce ──┘
                        │ rank 0
                        ▼
        checkpoint_round1/state.pkl  (shared /mnt/cluster_storage)
                        │
                        ▼  served in notebook 03
```

## Cell 1: Configuration

**What you do:** set the dataset/model repos, the shared-storage checkpoint
path, and the training hyper-parameters.

**What to check:** `MAX_TRAIN_STEPS = 50` keeps this a
demo-scale run that finishes in minutes. Set it to `None` for a full epoch.

**Why it matters:** every scale lever (steps, workers, batch) lives here. The
loop and infrastructure below never change.

In [ ]:
import logging, os, shutil, sys, time
from pathlib import Path
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s")
log = logging.getLogger("vla_finetune")

# --- All artifacts come from a PUBLIC S3 mirror (no Hugging Face at runtime) ------------
# Datasets + the PI0.5 model are mirrored to S3 so hundreds of concurrent clusters can't
# throttle HF. The ONLY gated dependency, the PaliGemma tokenizer, is baked into the
# image's HF cache -- HF_HUB_OFFLINE makes the preprocessor load it from there.
# LEROBOT_S3_ANON lets the datasource read the public bucket unsigned. Set these before
# importing the datasource; they're re-declared in ray.init so workers inherit them.
os.environ["LEROBOT_S3_ANON"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

MIRROR_ROOT     = "s3://anyscale-public-materials-use2/ray_summit_robotics_2026"
HF_DATASET_URI  = f"{MIRROR_ROOT}/libero"                 # LeRobot v3 dataset (streamed)
MODEL_S3_URI    = f"{MIRROR_ROOT}/pi05_libero_finetuned"  # PI0.5 model (staged per node)
BASE_S3_URI     = f"{MIRROR_ROOT}/pi05_base"              # base preprocessor config (tiny)
HF_PI05_REPO    = "lerobot/pi05_libero_finetuned"         # breadcrumb stored in checkpoints

# pi05_libero_finetuned: 7-D action, 8-D state, cameras image/image2 at 256x256.
LOCAL_MODEL_DIR      = Path("/mnt/local_storage/lerobot/pi05_libero_finetuned")
LOCAL_BASE_DIR       = Path("/mnt/local_storage/lerobot/pi05_base")
CLUSTER_STORAGE_ROOT = Path("/mnt/cluster_storage/vla_closed_loop_demo")  # shared FS
CAMERA_RENAME        = {}                       # LIBERO names need no remap

# PI0.5 is an action-CHUNKING policy: its config sets chunk_size == n_action_steps
# == 50, and the flow-matching loss compares v_t of shape [B, 50, action_dim]
# against u_t built from the target. Feeding one action per frame leaves u_t as
# [B, action_dim] and v_t as [B, 1, action_dim], and F.mse_loss BROADCASTS rather
# than raising -- so the model trains against a spurious axis and never learns
# chunks. The datasource assembles real chunks when told the size; it must match
# the policy config.
ACTION_CHUNK_SIZE = 50

MAX_TRAIN_STEPS = 50      # smoke run; set to None for a full epoch (~12 hrs)

CLUSTER_STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Dataset : {HF_DATASET_URI}")
print(f"Model   : {MODEL_S3_URI} -> {LOCAL_MODEL_DIR}")
print(f"Storage : {CLUSTER_STORAGE_ROOT}")

## Cell 2: Connect to Ray

**What you do:** connect and thread the environment every GPU worker needs.

**What to check:** the GPU count and node layout your cluster reports; the
training run below sizes itself to it (2 GPUs and 4 GPUs both work, on one node
or several). The env vars matter: `TORCHDYNAMO_DISABLE`
(PI0.5 calls `torch.compile`; workers have no C compiler), the `NCCL_*_DISABLE`
flags (force a safe TCP ring on containerized GPUs).

**Why it matters:** `working_dir="."` ships `tools/util.py` and `tools/lerobot_datasource.py`
to all workers. The env vars are the difference between a clean DDP run and a
segfault on this cluster image.

In [ ]:
import ray

ray.init(
    address="auto",
    runtime_env={
        "working_dir": ".",
        "env_vars": {
            "LEROBOT_S3_ANON":           "1",   # datasource reads the public bucket unsigned
            "HF_HUB_OFFLINE":            "1",    # PaliGemma tokenizer loads from the baked image cache
            "TRANSFORMERS_OFFLINE":      "1",
            "HF_HUB_ENABLE_HF_TRANSFER": "1",
            "PYTORCH_CUDA_ALLOC_CONF":   "expandable_segments:True",
            "TORCHDYNAMO_DISABLE":       "1",   # no C compiler on workers
            "NCCL_P2P_DISABLE":          "1",   # safe TCP ring on containerized GPUs
            "NCCL_SHM_DISABLE":          "1",
            "NCCL_IB_DISABLE":           "1",
        },
    },
    ignore_reinit_error=True,
)
logging.getLogger("ray.data").setLevel(logging.WARNING)
logging.getLogger("openlineage").setLevel(logging.WARNING)

# Cap what Ray Data may hold in the object store. Each worker node has 32 GB
# of host RAM, of which the raylet reserves ~10 GB for plasma; the video
# readers are ~4 orders of magnitude faster than one GPU training step, so
# without a limit the map output queue grows unboundedly, spills, and in the
# worst case the host OOM killer reaps the raylet (which kills the whole node).
# Default is unlimited.
ray.data.DataContext.get_current().execution_options.resource_limits = (
    ray.data.ExecutionResources.for_limits(object_store_memory=4 * 1024**3)
)

from tools import cluster
cluster.describe()
res = ray.cluster_resources()
print(f"Cluster: GPU={int(res['GPU'])}, CPU={int(res['CPU'])}, memory={res['memory']/1e9:.0f} GiB")
print("Ray Data object-store limit:",
      ray.data.DataContext.get_current().execution_options.resource_limits)


## Cell 3: Stage the model + open the data stream

**What you do:** stage the PI0.5 checkpoint to **each node's** local disk (it's
loaded from local safetensors by every worker), then open the LIBERO datasource
and extract the normalization `stats` the preprocessor needs.

**What to check:** staging prints `cached`/`downloaded` per node. The dataset
reports 1,693 episodes / 273k frames. **The dataset is never downloaded**. Only
the 3.4 GB model is staged and training data streams (notebook 01).

**Why it matters:** this is the scalable split: *artifacts* are staged and
reused per node and *data* is a stream. The same code path works whether LIBERO is
10 GB or 10 TB.

In [ ]:
from tools.viz import show_gif
show_gif("assets/nb02_cell3.gif",
         caption="stage 7.5 GB to each node's local disk once; then, open the stream (metadata only)")

from tools import util
from tools.lerobot_datasource import LeRobotDatasource

# Stage the PI0.5 model (7.5 GB) and the tiny base preprocessor config once per node,
# from the public S3 mirror (unsigned -- no HF token, no bucket credentials).
util.stage_on_all_nodes(
    ray, lambda: util.stage_model_to_local(MODEL_S3_URI, LOCAL_MODEL_DIR),
    MODEL_S3_URI, LOCAL_MODEL_DIR, log_fn=log.info,
)
util.stage_on_all_nodes(
    ray, lambda: util.stage_model_to_local(BASE_S3_URI, LOCAL_BASE_DIR),
    BASE_S3_URI, LOCAL_BASE_DIR, log_fn=log.info,
)

source       = LeRobotDatasource(HF_DATASET_URI, action_chunk_size=ACTION_CHUNK_SIZE)
TOTAL_FRAMES = source.meta.total_frames
STATS        = {
    k: {"mean": v["mean"], "std": v["std"]}
    for k, v in source.meta.stats.items()
    if k in ("action", "observation.state")
}
IMAGE_KEYS = [CAMERA_RENAME.get(k, k) for k in source.meta.video_keys]
print(f"Frames streaming: {TOTAL_FRAMES:,}  | cameras: {source.meta.video_keys}")

## Cell 4: The preprocessing pipeline (from notebook 01)

**What you do:** rebuild the lazy `read → rename → transpose` pipeline. This is
the same `build_libero_dataset` you stepped through in notebook 01.

**What to check:** it returns instantly (lazy). Nothing executes until the
trainer pulls shards.

**Why it matters:** the data layer is decoupled from training. The trainer just
asks for batches and Ray Data streams and preprocesses them on the CPU pool.

In [ ]:
# One DDP worker per GPU, read from the live cluster -- 2 GPUs on two nodes and
# 4 GPUs on one g6.12xlarge both work, and neither is written down anywhere here.
# Set NUM_WORKERS in the environment to pin a smaller run.
NUM_TRAIN_WORKERS = cluster.train_workers()


def rename_columns(row, rename):
    return {rename.get(k, k): v for k, v in row.items()}


def transpose_images(batch, camera_keys):
    """HWC -> CHW, staying uint8.

    The float32 cast belongs on the GPU, not here. As uint8 a 256x256x3 frame is
    197 KB; as float32 it is 786 KB. The datasource derives its block size from
    `estimated_row_size_bytes`, which counts video frames as uint8, so casting
    in-pipeline inflated every block 4x beyond what the streaming executor
    accounted for -- enough to push a 32 GB node into the OOM killer, which
    reaps the raylet and takes the node down with it.
    util.NumpyToTorchCollate(..., image_keys=...) does the widening on device.
    """
    out = dict(batch)
    for key in camera_keys:
        out[key] = np.transpose(np.stack(list(batch[key])), (0, 3, 1, 2))
    return out


def build_libero_dataset(max_steps=MAX_TRAIN_STEPS, batch_size=1,
                         num_workers=NUM_TRAIN_WORKERS):
    """Stream LIBERO frames, reading only as many as the step budget consumes.

    A 50-step smoke run at batch_size=1 across 2 workers touches ~100 rows out
    of 273,465 (more workers read proportionally more, since the limit below
    scales with num_workers). Streaming the full dataset anyway means the readers
    outrun the trainers by orders of magnitude: measured on the 2-GPU reference
    cluster, the map output queue reached 172 blocks / 21.6 GiB and spilled 10 GB
    to disk. `.limit()` pushes
    down into the reader so it simply stops early.

    MAX_TRAIN_STEPS = None (the full-epoch path) reads everything, as before.
    """
    ds = ray.data.read_datasource(source)
    if max_steps:
        # rows the loop consumes, plus one map batch of slack per worker
        ds = ds.limit(max_steps * batch_size * num_workers + 32 * num_workers)
    return (
        ds
        .map(rename_columns, fn_args=(CAMERA_RENAME,))
        .map_batches(transpose_images, batch_size=32, fn_args=(IMAGE_KEYS,))
    )


print(build_libero_dataset())


## Cell 5: The per-worker training loop

**What you do:** define `train_loop_per_worker`, the function Ray Train runs on
every GPU. It loads PI0.5 (freezing all but the action heads via
`util.load_pi05_policy`), wraps it with `prepare_model` (DDP), builds the
preprocessor, then iterates its **dataset shard** with gradient accumulation,
reporting metrics and a rank-0 checkpoint each epoch.

**What to check:** the three Ray Train touchpoints marked in comments are
`prepare_model` (DDP), `get_dataset_shard` (sharding), `train.report`
(metrics + checkpoint). Everything else is ordinary PyTorch.

**Why it matters:** this is the whole point: a normal loop becomes a
distributed, fault-tolerant, checkpointing 4-GPU job with three Ray calls.

In [ ]:
import ray.train
import ray.train.torch


def train_loop_per_worker(config):
    device = torch.device("cuda")

    # Stage the model onto THIS node before loading it. stage_model_to_local is
    # idempotent (it returns early when config.json is already there), so this
    # is a no-op on nodes the driver already staged. It exists because
    # stage_on_all_nodes() only reaches nodes that are alive when it runs: a
    # worker placed on a node the autoscaler added afterwards finds no local
    # model dir, and lerobot's PreTrainedConfig.from_pretrained then falls
    # through to hf_hub_download() and dies with
    #   HFValidationError: Repo id must be in the form 'repo_name' ...:
    #   '/mnt/local_storage/lerobot/pi05_libero_finetuned'
    # which is what the 8 preceding WorkerGroupStartupTimeouts were a symptom of.
    util.stage_model_to_local(config["model_uri"], config["model_dir"])
    util.stage_model_to_local(config["base_uri"], config["base_dir"])

    policy = util.load_pi05_policy(config["model_dir"])
    policy = ray.train.torch.prepare_model(policy)              # <-- RAY TRAIN: DDP wrap

    optimizer = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=config.get("lr", 5e-5),
    )
    scaler = torch.amp.GradScaler("cuda")

    checkpoint = ray.train.get_checkpoint()                     # <-- RAY TRAIN: fault tolerance
    start_epoch, step = (util.load_checkpoint(checkpoint, policy, optimizer, scaler)
                         if checkpoint else (0, 0))

    # Base preprocessor config comes from the S3 mirror, staged above.
    # (It reads only the JSON here; the gated PaliGemma tokenizer it references
    #  loads from the image's baked HF cache under HF_HUB_OFFLINE=1.)
    # util.build_preprocessor pins the pipeline's DeviceProcessorStep to cuda --
    # the saved pi05_base JSON says "cpu" and does NOT inherit policy.config.device,
    # so the default pipeline hauls every batch back to the host and the CUDA
    # model then meets CPU token ids ("two devices, cuda:0 and cpu", index_select).
    preprocessor, _ = util.build_preprocessor(
        policy.module.config, config["base_dir"], config["stats"], device=device,
    )

    batch_size      = int(config.get("batch_size", 1))
    grad_accum      = int(config.get("grad_accum", 8))
    num_epochs      = int(config.get("num_epochs", 1))
    max_len         = int(config.get("max_len", 512))
    max_train_steps = config.get("max_train_steps")
    num_workers     = ray.train.get_context().get_world_size()
    rank            = ray.train.get_context().get_world_rank()
    scheduler       = util.build_lr_scheduler(optimizer, config, num_workers, last_step=step)
    shard           = ray.train.get_dataset_shard("train")      # <-- RAY DATA: per-worker shard

    # image_keys tells the collate which columns arrive as uint8 and must be
    # widened to float32 here on the GPU rather than back in the pipeline.
    collate = util.NumpyToTorchCollate(device, image_keys=config["image_keys"])

    # A restored checkpoint from a completed run makes this loop range(1, 1) --
    # zero iterations, no ray.train.report, and fit() then returns the PREVIOUS
    # run's checkpoint and metrics as if they were fresh. Say so instead.
    if util.resume_would_skip_training(start_epoch, num_epochs):
        log.warning(
            "RESUMED A COMPLETED RUN: checkpoint is at epoch %d and num_epochs=%d, "
            "so NO training steps will run. Any metrics below are from the earlier "
            "run. To retrain, raise num_epochs or change RunConfig(name=...).",
            start_epoch - 1, num_epochs,
        )
        ray.train.report({"epoch": start_epoch - 1, "steps": step, "loss": float("nan"),
                          "lr": float("nan"), "skipped_resumed_complete": True})
        return

    for epoch in range(start_epoch, num_epochs):
        optimizer.zero_grad(set_to_none=True)
        accum = 0
        loss_sum, loss_count = 0.0, 0

        for batch in shard.iter_torch_batches(batch_size=batch_size, collate_fn=collate):
            loss_val = util.train_step(policy, batch, preprocessor, max_len, grad_accum, scaler)
            step += 1; accum += 1
            loss_sum += loss_val; loss_count += 1

            if accum % grad_accum == 0:
                util.optimizer_step(policy, optimizer, scaler, scheduler)
                accum = 0

            if step % 10 == 0 and rank == 0:
                log.info("epoch=%d  step=%d  loss=%.4f  lr=%.2e  gpu_peak=%.1f GB",
                         epoch, step, loss_val, scheduler.get_last_lr()[0],
                         torch.cuda.max_memory_allocated() / 1e9)

            if max_train_steps and step >= max_train_steps:
                break

        if accum > 0:
            util.optimizer_step(policy, optimizer, scaler, scheduler)

        avg_loss = loss_sum / max(loss_count, 1)
        metrics  = {"epoch": epoch, "steps": step,
                    "loss": avg_loss, "lr": scheduler.get_last_lr()[0],
                    "gpu_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}

        if rank == 0:
            ckpt = util.make_checkpoint(
                policy, optimizer, scaler, epoch, step, config["stats"],
                base_model_repo=HF_PI05_REPO, camera_rename=CAMERA_RENAME,
            )
            ray.train.report(metrics, checkpoint=ckpt)          # <-- RAY TRAIN: synced report
        else:
            ray.train.report(metrics)

        if max_train_steps and step >= max_train_steps:
            break


## Cell 6: Launch helper

**What you do:** wrap `TorchTrainer` in `run_training`, which sets the
`ScalingConfig` (one GPU worker per GPU), `RunConfig` (storage + `FailureConfig` +
checkpoint retention), passes the dataset, calls `.fit()`, then copies the
resulting checkpoint to a **stable path** (`checkpoint_round1/state.pkl`) so the
serving replica in notebook 03 can read it without knowing Ray Train's internal
layout.

**What to check:** `num_workers=NUM_TRAIN_WORKERS, use_gpu=True`, where
`NUM_TRAIN_WORKERS` is the cluster's GPU count. The effective batch is
`batch_size(1) × grad_accum(16) × num_workers`, which is 32 on 2 GPUs and 64 on 4. To hold
the effective batch constant across cluster sizes, scale `grad_accum` down as
workers go up.

**Why it matters:** scaling to more GPUs is editing `num_workers`. No additional changes are needed.

In [ ]:
def run_training(ds, round_name):
    """Run TorchTrainer on `ds`, copy checkpoint to a stable path, return path + metrics."""
    CLUSTER_STORAGE_ROOT.mkdir(parents=True, exist_ok=True)

    # Ray Train RESTORES any existing run with this name. If that run already
    # finished its epochs, the train loop body never executes and fit() returns
    # the OLD checkpoint and metrics -- so the "Final metrics" printed below
    # would describe training that did not happen. Warn on the driver, where
    # that output is actually read.
    prior_run = CLUSTER_STORAGE_ROOT / f"vla-finetune-{round_name}"
    if prior_run.exists() and any(prior_run.glob("checkpoint_*")):
        log.warning(
            "run %r already exists at %s -- Ray Train will RESTORE it, not retrain. "
            "If it completed its epochs, the metrics and checkpoint below come from "
            "that earlier run. Delete that directory or use a new round_name to "
            "train from scratch.", prior_run.name, prior_run,
        )



    # Ray Train places one worker per node, and loading PI0.5 spikes host RAM to
    # ~16 GB transiently (steady state is only ~3.3 GB). A node left holding idle
    # spill workers from an earlier stage can have as little as ~12 GB free, and
    # the worker is then killed mid-load:
    #   Worker exit type: SYSTEM_ERROR
    #   Worker unexpectedly exits with a connection error code 2. End of file.
    # That is the host OOM killer, surfaced as a worker health-check failure.
    # require_all=True because EVERY node will host a worker, not just one.
    util.release_phase(ray, log_fn=log.info)
    util.wait_for_host_headroom(ray, need_mib=17_000, require_all=True,
                            num_workers=NUM_TRAIN_WORKERS, log_fn=log.info)

    result = ray.train.torch.TorchTrainer(
        train_loop_per_worker=train_loop_per_worker,
        train_loop_config={
                "stats":           STATS,
                "total_rows":      TOTAL_FRAMES,
                "num_epochs":      1,
                "batch_size":      1,      # measured peak 8.9 GB at bs=1, fits a 16 GB GPU
                "grad_accum":      16,     # effective batch = 1 * 16 * NUM_TRAIN_WORKERS
                "lr":              5e-5,
                "warmup_frac":     0.1,
                "max_len":         512,
                "max_train_steps": MAX_TRAIN_STEPS,
                # camera columns stay uint8 through Ray Data; the collate widens
                # them to float32 on the GPU
                "image_keys":      IMAGE_KEYS,
                # each worker stages the model to its own node's local disk
                "model_uri":       MODEL_S3_URI,
                "base_uri":        BASE_S3_URI,
                "model_dir":       LOCAL_MODEL_DIR,
                "base_dir":        LOCAL_BASE_DIR,
            },
        scaling_config=ray.train.ScalingConfig(num_workers=NUM_TRAIN_WORKERS, use_gpu=True),
        run_config=ray.train.RunConfig(
            name=f"vla-finetune-{round_name}",
            storage_path=str(CLUSTER_STORAGE_ROOT),
            failure_config=ray.train.FailureConfig(max_failures=1),
            checkpoint_config=ray.train.CheckpointConfig(num_to_keep=1),
        ),
        datasets={"train": ds},
    ).fit()

    checkpoint_path = CLUSTER_STORAGE_ROOT / f"checkpoint_{round_name}" / "state.pkl"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    with result.checkpoint.as_directory() as d:
        shutil.copy2(os.path.join(d, "state.pkl"), checkpoint_path)

    log.info("[%s] checkpoint -> %s", round_name, checkpoint_path)
    log.info("[%s] metrics: %s", round_name, result.metrics)
    return checkpoint_path, result.metrics


## Cell 7: Fine-tune PI0.5 on every GPU in the cluster

**What you do:** launch the distributed run. Ray Train spins up one GPU worker
per GPU,
each loads PI0.5, wraps it in DDP, and trains on its shard of the LIBERO stream.

**What to check:** `(RayTrainWorker ...)` log lines show `epoch=0 step=N
loss=...` from rank 0 every 10 steps, then a checkpoint is written. With
`MAX_TRAIN_STEPS=50` this is a couple of minutes after model load.

**Why it matters:** this is the full distributed fine-tune. Every later
notebook reuses this exact `TorchTrainer` pattern.

> **Re-running note:** Ray Train resumes from `storage_path` when a run of the
> same `name` already has a checkpoint there. On a fresh cluster this trains
> from scratch (as shown below). To force a fresh run on a populated path,
> clear that run directory first.

In [ ]:
from tools.viz import show_gif
show_gif("assets/nb02_cell7.gif",
         caption="4 DDP workers x LIBERO shards: 50 steps, all-reduce every step, checkpoint to shared storage")
         
ckpt_path, metrics = run_training(build_libero_dataset(), "round1")
print("\nCheckpoint :", ckpt_path)
print("Final metrics:", metrics)

## Cell 8: Confirm the checkpoint

**What you do:** verify the checkpoint landed on shared storage and peek at its
contents.

**What to check:** `state.pkl` exists and contains the trained head weights
plus the dataset `stats` (so the serving preprocessor can be rebuilt) and
breadcrumbs (`base_model_repo`, `step`, `epoch`). Note we load it with a
CPU-mapping unpickler: training ran on GPU workers so the tensors are tagged
CUDA, but this notebook's kernel runs on the CPU-only head node.

**Why it matters:** this small, self-describing checkpoint is the hand-off to
notebook 03. The Ray Serve replica loads it onto a GPU and serves predictions
over HTTP.

In [ ]:
import io, pickle, torch


class _CPUUnpickler(pickle.Unpickler):
    """Load a CUDA-saved checkpoint on the CPU-only head node.

    The trainer ran on GPU workers, so the checkpoint's tensors are tagged
    CUDA. This notebook's kernel is on the CPU head, so we remap storages to
    CPU on load, the same trick tools/policy_server.py uses to inspect checkpoints.
    """
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu", weights_only=False)
        return super().find_class(module, name)


print("exists:", ckpt_path.exists(), "| size:", f"{ckpt_path.stat().st_size/1e6:.1f} MB")
with open(ckpt_path, "rb") as f:
    state = _CPUUnpickler(f).load()
print("keys           :", list(state.keys()))
print("trained tensors:", len(state["model"]))
print("stats keys     :", list(state["stats"].keys()))
print("step / epoch   :", state["step"], "/", state["epoch"])

## Conclusion

You fine-tuned a 3.4B-parameter PI0.5 policy across every GPU in the cluster
with **Ray Train:** DDP via `prepare_model`, streaming shards via
`get_dataset_shard`, and a
fault-tolerant checkpoint via `train.report`, all wrapped around an ordinary
PyTorch loop. Only the action heads were trained. The checkpoint is small and
self-describing.

**Ray primitives used:** `TorchTrainer`, `prepare_model`, `get_dataset_shard`,
`train.report`, `ScalingConfig`, `FailureConfig`, `CheckpointConfig`.

**Scaling levers:** `ScalingConfig(num_workers=N)` (2 → 400 GPUs);
`MAX_TRAIN_STEPS=None` (full epoch); larger `batch_size` on bigger GPUs.

Next, **`03_serving_and_sim_eval.ipynb`** loads this checkpoint into **Ray
Serve**, then fans out **Isaac Lab** simulation rollouts as Ray tasks that query
the policy over HTTP, and closes the loop by folding the sim data back into
training.